In [ ]:
import os

# Your repository details
REPO_URL = "https://github.com/ninielilililili/css314_230103071_week6-10.git"
REPO_NAME = "css314_230103071_week6-10"
STUDENT_ID = "230103071"

# Clone if not already cloned
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

# Create a new subfolder/directory for Lab 02 files
NEW_DIR = f"lab02_23103071_practicehour"
os.makedirs(NEW_DIR, exist_ok=True)
%cd {NEW_DIR}

print(f"Current Working Directory: {os.getcwd()}")

Cloning into 'css314_230103071_week6-10'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 37.40 KiB | 18.70 MiB/s, done.
/content/css314_230103071_week6-10
/content/css314_230103071_week6-10/lab02_23103071_practicehour
Current Working Directory: /content/css314_230103071_week6-10/lab02_23103071_practicehour


In [ ]:
%%writefile task1_divergence.py
import math
import time
import numpy as np
from numba import cuda

N = 1 << 20  # 1,048,576
ITERATIONS = 1000

@cuda.jit
def kernel_a_uniform(y):
    idx = cuda.grid(1)
    if idx < y.size:
        val = y[idx]
        for _ in range(ITERATIONS):
            val = val * 1.0001 + 0.0001
        y[idx] = val

@cuda.jit
def kernel_b_divergent(y):
    idx = cuda.grid(1)
    if idx < y.size:
        val = y[idx]
        if idx % 2 == 0:
            for _ in range(ITERATIONS):
                val = val * 1.0001 + 0.0001
        else:
            for _ in range(ITERATIONS):
                val = (val - 0.0001) / 1.0001
        y[idx] = val

@cuda.jit
def kernel_c_warp_aligned(y):
    idx = cuda.grid(1)
    if idx < y.size:
        val = y[idx]
        warp_id = idx // 32
        if warp_id % 2 == 0:
            for _ in range(ITERATIONS):
                val = val * 1.0001 + 0.0001
        else:
            for _ in range(ITERATIONS):
                val = (val - 0.0001) / 1.0001
        y[idx] = val

def benchmark_kernel(kernel_func, h_data):
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    d_data = cuda.to_device(h_data)

    # Warm-up launch
    kernel_func[blocks_per_grid, threads_per_block](d_data)
    cuda.synchronize()

    times = []
    for _ in range(10):
        d_data.copy_to_device(h_data)
        cuda.synchronize()

        start = time.perf_counter()
        kernel_func[blocks_per_grid, threads_per_block](d_data)
        cuda.synchronize()
        end = time.perf_counter()

        times.append((end - start) * 1000.0)

    return float(np.mean(times))

if __name__ == "__main__":
    h_data = np.ones(N, dtype=np.float32)

    time_a = benchmark_kernel(kernel_a_uniform, h_data)
    time_b = benchmark_kernel(kernel_b_divergent, h_data)
    time_c = benchmark_kernel(kernel_c_warp_aligned, h_data)

    print(f"Kernel A (Uniform Path):        {time_a:.3f} ms")
    print(f"Kernel B (Full Divergence):     {time_b:.3f} ms")
    print(f"Kernel C (Warp-Aligned Branch): {time_c:.3f} ms")

Writing task1_divergence.py


In [ ]:
%%writefile task2_stencil_1d.py
import numpy as np
from numba import cuda

@cuda.jit
def stencil_1d(d_in, d_out, N):
    idx = cuda.grid(1)
    if idx < N:
        if idx == 0:
            left = d_in[0]
            right = d_in[1] if N > 1 else d_in[0]
        elif idx == N - 1:
            left = d_in[N - 2] if N > 1 else d_in[N - 1]
            right = d_in[N - 1]
        else:
            left = d_in[idx - 1]
            right = d_in[idx + 1]

        d_out[idx] = 0.25 * left + 0.5 * d_in[idx] + 0.25 * right

def run_stencil(h_in):
    N = len(h_in)
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array(N, dtype=np.float32)

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    stencil_1d[blocks_per_grid, threads_per_block](d_in, d_out, N)
    cuda.synchronize()

    return d_out.copy_to_host()

def cpu_stencil(arr):
    padded = np.pad(arr, (1, 1), mode='edge')
    return 0.25 * padded[:-2] + 0.5 * padded[1:-1] + 0.25 * padded[2:]

if __name__ == "__main__":
    N = 100007
    test_in = np.sin(np.linspace(0, 10, N)).astype(np.float32)

    h_gpu = run_stencil(test_in)
    h_cpu = cpu_stencil(test_in)

    max_delta = np.max(np.abs(h_gpu - h_cpu))
    assert np.allclose(h_gpu, h_cpu, atol=1e-4)
    print(f"TASK 2 PASSED: MAX DELTA = {max_delta:.6e}")

Writing task2_stencil_1d.py


In [ ]:
%%writefile task3_grid_stride.py
import numpy as np
from numba import cuda

@cuda.jit
def grid_stride_scale_kernel(d_arr, factor, N):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)

    for i in range(start, N, stride):
        d_arr[i] = d_arr[i] * factor

def run_grid_stride(h_arr, factor):
    N = len(h_arr)
    d_arr = cuda.to_device(h_arr)

    threads_per_block = 256
    blocks_per_grid = 64

    grid_stride_scale_kernel[blocks_per_grid, threads_per_block](d_arr, factor, N)
    cuda.synchronize()

    return d_arr.copy_to_host()

if __name__ == "__main__":
    N = 16777216  # 2^24
    factor = 4.25
    h_arr = np.ones(N, dtype=np.float32)

    res = run_grid_stride(h_arr, factor)

    assert np.allclose(res, factor)
    print(f"TASK 3 PASSED: All {N} elements successfully scaled by {factor}")

Writing task3_grid_stride.py


In [ ]:
%%writefile task4_sobel_2d.py
import math
import numpy as np
from numba import cuda

@cuda.jit
def sobel_x_kernel(d_in, d_out, rows, cols):
    col, row = cuda.grid(2)

    if row < rows and col < cols:
        if row == 0 or row == rows - 1 or col == 0 or col == cols - 1:
            d_out[row, col] = 0.0
        else:
            d_out[row, col] = (
                -1.0 * d_in[row - 1, col - 1] + 1.0 * d_in[row - 1, col + 1]
                - 2.0 * d_in[row, col - 1]     + 2.0 * d_in[row, col + 1]
                - 1.0 * d_in[row + 1, col - 1] + 1.0 * d_in[row + 1, col + 1]
            )

def run_sobel(h_img):
    rows, cols = h_img.shape
    d_in = cuda.to_device(h_img)
    d_out = cuda.device_array((rows, cols), dtype=np.float32)

    threads_2d = (16, 16)
    blocks_x = math.ceil(cols / threads_2d[0])
    blocks_y = math.ceil(rows / threads_2d[1])
    grid_2d = (blocks_x, blocks_y)

    sobel_x_kernel[grid_2d, threads_2d](d_in, d_out, rows, cols)
    cuda.synchronize()

    return d_out.copy_to_host()

if __name__ == "__main__":
    h_img = np.ones((2048, 2048), dtype=np.float32)
    sobel_res = run_sobel(h_img)

    assert np.max(np.abs(sobel_res[1:-1, 1:-1])) < 1e-5
    assert np.all(sobel_res[0, :] == 0.0)
    assert np.all(sobel_res[:, 0] == 0.0)
    print("TASK 4 PASSED: 2D Horizontal Sobel Kernel output verified.")

Writing task4_sobel_2d.py


In [ ]:
%%writefile verify_submission.py
import sys
import hashlib
import numpy as np
from numba import cuda

def main():
    print("=" * 60)
    print("RUNNING CUDA LAB 02 AUTONOMOUS VERIFICATION")
    print("=" * 60)

    student_id = "230103071"

    # 1. Verify Task 2
    try:
        import task2_stencil_1d as t2
        N = 100007
        test_in = np.sin(np.linspace(0, 10, N)).astype(np.float32)
        h_gpu = t2.run_stencil(test_in)
        h_cpu = t2.cpu_stencil(test_in)
        assert np.allclose(h_gpu, h_cpu, atol=1e-4), "Task 2 output mismatch against CPU"
        print("[PASS] Task 2 (1D Stencil & Clamping)")
    except Exception as e:
        print(f"[FAIL] Task 2: {e}")
        sys.exit(1)

    # 2. Verify Task 3
    try:
        import task3_grid_stride as t3
        N = 100000
        test_arr = np.ones(N, dtype=np.float32)
        factor = 4.25
        res = t3.run_grid_stride(test_arr, factor)
        assert np.allclose(res, factor), "Task 3 elements not uniformly scaled"
        print("[PASS] Task 3 (Grid-Stride Scaling)")
    except Exception as e:
        print(f"[FAIL] Task 3: {e}")
        sys.exit(1)

    # 3. Verify Task 4
    try:
        import task4_sobel_2d as t4
        test_img = np.ones((64, 64), dtype=np.float32)
        sobel_res = t4.run_sobel(test_img)
        assert np.max(np.abs(sobel_res[1:-1, 1:-1])) < 1e-5, "Task 4 interior gradient != 0"
        assert np.all(sobel_res[0, :] == 0.0), "Task 4 border rows not zeroed"
        assert np.all(sobel_res[:, 0] == 0.0), "Task 4 border columns not zeroed"
        print("[PASS] Task 4 (2D Sobel Horizontal)")
    except Exception as e:
        print(f"[FAIL] Task 4: {e}")
        sys.exit(1)

    # Token Generation
    hasher = hashlib.sha256()
    hasher.update(student_id.encode('utf-8'))
    try:
        device_name = cuda.get_current_device().name
        hasher.update(device_name if isinstance(device_name, bytes) else device_name.encode('utf-8'))
    except Exception:
        hasher.update(b"UNKNOWN_CUDA_DEVICE")

    hasher.update(h_gpu[:32].tobytes())
    hasher.update(sobel_res[:8, :8].tobytes())
    token = hasher.hexdigest()[:20].upper()

    print("\n" + "=" * 60)
    print(f"VERIFICATION SUCCESSFUL")
    print(f"OFFICIAL SUBMISSION TOKEN: {token}")
    print("=" * 60)

if __name__ == "__main__":
    main()

Writing verify_submission.py
Overwriting verify_submission.py


In [ ]:
# Run Task 1 benchmark
!python task1_divergence.py

print("\n Running Verification Script...\n")

# Run verification script
!python verify_submission.py

Kernel A (Uniform Path):        23.775 ms
Kernel B (Full Divergence):     92.860 ms
Kernel C (Warp-Aligned Branch): 46.513 ms

 Running Verification Script...

RUNNING CUDA LAB 02 AUTONOMOUS VERIFICATION
[PASS] Task 2 (1D Stencil & Clamping)
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
[PASS] Task 3 (Grid-Stride Scaling)
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
[PASS] Task 4 (2D Sobel Horizontal)

VERIFICATION SUCCESSFUL
OFFICIAL SUBMISSION TOKEN: C56E46B2C5AE3124CF9A


%%writefile README.md
# CUDA Lab 02: Advanced Geometries & Stencils

**Student ID:** 230103071
**Allocated GPU Node:** Tesla T4
**CUDA Compute Capability:** 7.5
**Official Verification Token:** C56E46B2C5AE3124CF9A

## Task 1: Warp Divergence Benchmarks

| Kernel | Description | Execution Time (ms) |
| :--- | :--- | :--- |
| Kernel A | Uniform Path (No Divergence) | 23.775 ms |
| Kernel B | Interleaved Thread Divergence (`idx % 2`) | 92.860 ms |
| Kernel C | Warp-Aligned Branching (`warp_id % 2`) | 46.513 ms |

### Analysis
- **Kernel A vs. Kernel B:** Kernel B executes significantly slower (92.860 ms vs 23.775 ms) because alternating thread conditions (`idx % 2`) force threads within the same 32-thread warp to diverge. The GPU hardware serializes the two execution paths, halving instruction throughput.
- **Kernel C vs. Kernel B:** Kernel C improves performance over Kernel B (46.513 ms vs 92.860 ms) because branching is aligned by warp ID (`warp_id % 2`). All 32 threads within any given warp execute the exact same instruction path without intra-warp divergence.